In [ ]:
!pip install sparqlwrapper

In [5]:
import sys
from pandas import json_normalize
import pandas as pd
import json
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint_url = "https://query.wikidata.org/sparql"

# Get cities
query = """SELECT DISTINCT ?grandeville ?grandevilleLabel ?pays ?paysLabel ?image {
  ?grandeville wdt:P31 wd:Q1549591;
               wdt:P17 ?pays;
               wdt:P18 ?image.
 SERVICE wikibase:label { bd:serviceParam wikibase:language "fr". }
}
LIMIT 5"""


def get_results(endpoint_url, query):
    user_agent = "WDQS-example Python/%s.%s" % (
        sys.version_info[0],
        sys.version_info[1],
    )
    sparql = SPARQLWrapper(endpoint_url, agent=user_agent)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.query().convert()


array = []
results = get_results(endpoint_url, query)

for result in results["results"]["bindings"]:
    array.append(
        (
            result["grandevilleLabel"]["value"],
            result["paysLabel"]["value"],
            result["image"]["value"],
        )
    )


dataframe = pd.DataFrame(array, columns=["ville", "pays", "image"])
dataframe = dataframe.astype(
    dtype={"ville": "<U200", "pays": "<U200", "image": "<U200"}
)
dataframe

,ville,pays,image
0,Dakar,Sénégal,http://commons.wikimedia.org/wiki/Special:File...
1,Manama,Bahreïn,http://commons.wikimedia.org/wiki/Special:File...
2,Tripoli,Libye,http://commons.wikimedia.org/wiki/Special:File...
3,Dodoma,Tanzanie,http://commons.wikimedia.org/wiki/Special:File...
4,Asmara,Érythrée,http://commons.wikimedia.org/wiki/Special:File...


## Téléchargement des images sur le dossier du projet

In [8]:
import requests
import shutil
import os

def download_image(url, folder="images"):
    """Télécharge une image depuis une URL et l'enregistre dans un dossier spécifique."""
    
    # Crée le dossier s'il n'existe pas
    os.makedirs(folder, exist_ok=True)

    # Récupère le nom du fichier depuis l'URL
    filename = os.path.join(folder, os.path.basename(url))

    headers = {"User-Agent": "Mozilla/5.0"}
    request = requests.get(url, allow_redirects=True, headers=headers, stream=True)

    if request.status_code == 200:
        with open(filename, "wb") as image:
            request.raw.decode_content = True
            shutil.copyfileobj(request.raw, image)

    return request.status_code

# Application sur ton dataframe
dataframe.image.apply(lambda url: download_image(url, folder="images"))


0    200
1    200
2    200
3    200
4    200
Name: image, dtype: int64